<a href="https://colab.research.google.com/github/faculatini/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faculatini/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [24]:
!git clone https://github.com/faculatini/flyrank_ml.git

Cloning into 'flyrank_ml'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 140 (delta 50), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.84 MiB | 11.16 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [25]:
%cd flyrank_ml

/content/flyrank_ml/flyrank_ml


In [26]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 30,000
Columns: 44


In [27]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClients:", df["client_id"].nunique())
print("Content items:", df["content_id"].nunique())

print("\nDuplicate content_id rows:", df["content_id"].duplicated().sum())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Clients: 32
Content items: 30000

Duplicate content_id rows: 0


In [28]:
signal_columns = [
    "days_since_last_update",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "trend_direction",
    "trend_pct",
]

df[signal_columns].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
days_since_last_update,30000.0,NaN,NaN,NaN,46.0983,42.078709,1.0,20.0,20.0,104.0,373.0
content_age_days,30000.0,NaN,NaN,NaN,256.1678,132.70793,90.0,132.0,236.0,333.0,564.0
impressions_90d,30000.0,NaN,NaN,NaN,5200.3663,16838.019547,1.0,81.0,731.0,3615.25,517715.0
clicks_90d,30000.0,NaN,NaN,NaN,16.097333,75.076958,0.0,0.0,1.0,7.0,4178.0
ctr,30000.0,NaN,NaN,NaN,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,NaN,NaN,NaN,16.34238,15.21679,0.0,6.2,10.8,22.3,245.0
sessions_90d,30000.0,NaN,NaN,NaN,37.066633,107.069131,1.0,2.0,7.0,27.0,4345.0
engagement_rate,30000.0,NaN,NaN,NaN,2.53452,8.310096,0.0,0.0,0.0,1.35,100.0
trend_direction,30000,5,down,16262,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trend_pct,26612.0,NaN,NaN,NaN,-4.785969,473.86178,-100.0,-62.6,-33.5,0.0,44900.0


In [29]:
print("avg_position == 0:", (df["avg_position"] == 0).sum())
print("Missing ctr:", df["ctr"].isna().sum())
print("Missing impressions:", df["impressions_90d"].isna().sum())
print("Missing sessions:", df["sessions_90d"].isna().sum())
print("Missing days_since_last_update:", df["days_since_last_update"].isna().sum())

print("\nTrend direction:")
print(df["trend_direction"].value_counts(dropna=False))

avg_position == 0: 1205
Missing ctr: 0
Missing impressions: 0
Missing sessions: 0
Missing days_since_last_update: 0

Trend direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks and rule reasoning

Before encoding the baseline rule, I will check whether the signals behind the rule are actually visible in the data.

Each signal check uses a human-readable bucket table with:

* the signal bucket;
* the number of pages in the bucket (`n`);
* the observed outcome rate.

The verdict for each signal will be exactly one of:

* `CONFIRMED` — the expected pattern is clearly visible;
* `OPPOSITE` — the observed pattern points in the opposite direction;
* `MIXED` — the pattern appears only in part of the buckets or is not strong enough to stand alone;
* `FALSE` — the expected pattern is not visible.

A negative or mixed result is useful: it prevents a weak assumption from becoming part of the baseline rule.

At least one of the two checked signals must be connected to a real FlyRank flag discussed in the Week-4 session. The final rule will only use observable inputs available at decision time and will not use the decline label, `trend_direction`, or `trend_pct` as features.


### Signal check 1 — Staleness

**FlyRank flag connection:** refresh flags.

**Belief being tested:** pages that have gone longer without an update should show a higher rate of decline.

I will use the predefined `freshness_tier` buckets from the dataset so that the comparison remains transparent and reproducible. The outcome being audited is the observed declining rate (`is_declining_label`), which is used here only as the outcome of the signal check and not as an input to the baseline rule.

The table will report both the declining rate and `n` for every bucket.


In [30]:
staleness_check = (
    df.groupby("freshness_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda s: (s == "down").mean() * 100)
      )
      .reset_index()
)

freshness_order = ["never", "0-30", "31-90", "91-180", "181+"]

staleness_check["freshness_tier"] = pd.Categorical(
    staleness_check["freshness_tier"],
    categories=freshness_order,
    ordered=True
)

staleness_check = (
    staleness_check
    .sort_values("freshness_tier")
    .assign(
        declining_rate=lambda x: x["declining_rate"].round(1)
    )
)

staleness_check


,freshness_tier,n,declining_rate
0,0-30,20480,51.1
2,31-90,175,58.9
3,91-180,9171,61.1
1,181+,174,47.1


The observed pattern is not monotonic. The declining rate increases from 51.1% for pages updated within 30 days to 61.1% for pages in the 91–180 day bucket, but then falls to 47.1% for pages older than 180 days.

The two extreme buckets also have small sample sizes (n=175 and n=174), so their rates should not be treated as strong evidence on their own.

Verdict: MIXED

Staleness provides useful evidence for prioritization, but it is not sufficient to justify a rule that assigns increasing priority solely as pages become older.

### Signal check 2 — Visibility

**FlyRank flag connection:** quick-win / volume logic.

**Belief being tested:** pages with more search visibility provide stronger evidence and represent a larger opportunity when they are declining.

I will use the predefined `impression_tier` buckets from the dataset. The purpose of this check is not to claim that high-impression pages are inherently worse. Instead, it tests whether declining pages are distributed differently across visibility levels and whether visibility provides useful context for prioritization.

The table will report the declining rate and `n` for every bucket.


In [31]:
volume_check = (
    df.groupby("impression_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda s: (s == "down").mean() * 100)
      )
      .reset_index()
)

impression_order = [
    "no_data",
    "none",
    "low",
    "moderate",
    "good",
    "excellent"
]

volume_check["impression_tier"] = pd.Categorical(
    volume_check["impression_tier"],
    categories=impression_order,
    ordered=True
)

volume_check = (
    volume_check
    .sort_values("impression_tier")
    .assign(
        declining_rate=lambda x: x["declining_rate"].round(1)
    )
)

volume_check

,impression_tier,n,declining_rate
2,low,11248,45.4
3,moderate,10469,61.5
1,good,7205,58.6
0,excellent,1078,46.2


The observed declining rate is highest in the moderate and good visibility buckets, while both low and excellent visibility have lower declining rates. The relationship is therefore not monotonic.

Verdict: MIXED

Visibility is useful context for prioritization, but higher impressions should not be interpreted as a stronger decline signal by itself. The baseline should treat visibility as context or opportunity size rather than assuming that more impressions automatically means greater decline risk.

In [32]:
combined_signal_check = (
    df.groupby(["freshness_tier", "impression_tier"], dropna=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda s: (s == "down").mean() * 100)
      )
      .reset_index()
)

combined_signal_check["declining_rate"] = (
    combined_signal_check["declining_rate"].round(1)
)

combined_signal_check.sort_values(
    ["freshness_tier", "impression_tier"]
)

,freshness_tier,impression_tier,n,declining_rate
0,0-30,excellent,582,48.5
1,0-30,good,4149,57.8
2,0-30,low,9082,42.4
3,0-30,moderate,6667,59.1
4,181+,excellent,2,100.0
5,181+,good,7,100.0
6,181+,low,152,42.1
7,181+,moderate,13,69.2
8,31-90,excellent,7,42.9
9,31-90,good,21,38.1


In [33]:
combined_signal_check[
    combined_signal_check["n"] >= 100
].sort_values(
    "declining_rate",
    ascending=False
)

,freshness_tier,impression_tier,n,declining_rate
15,91-180,moderate,3697,65.6
13,91-180,good,3028,59.8
3,0-30,moderate,6667,59.1
14,91-180,low,1959,59.0
1,0-30,good,4149,57.8
0,0-30,excellent,582,48.5
12,91-180,excellent,487,43.3
2,0-30,low,9082,42.4
6,181+,low,152,42.1


In [34]:
moderate_visibility = df[
    (df["freshness_tier"] == "91-180") &
    (df["impression_tier"] == "moderate")
].copy()

print("Rows:", len(moderate_visibility))

moderate_visibility["impression_bucket"] = pd.cut(
    moderate_visibility["impressions_90d"],
    bins=[299, 500, 750, 1000, 1500, 2000, 2999],
    labels=[
        "300-500",
        "501-750",
        "751-1000",
        "1001-1500",
        "1501-2000",
        "2001-2999"
    ]
)

moderate_detail = (
    moderate_visibility
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        declining_rate=(
            "trend_direction",
            lambda s: (s == "down").mean() * 100
        ),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

moderate_detail["declining_rate"] = moderate_detail["declining_rate"].round(1)

moderate_detail

Rows: 3697


,impression_bucket,n,declining_rate,median_impressions
0,300-500,657,62.9,394.0
1,501-750,617,63.0,622.0
2,751-1000,471,67.5,868.0
3,1001-1500,667,68.2,1205.0
4,1501-2000,509,63.5,1737.0
5,2001-2999,776,68.2,2449.5


### Baseline rule

The signal checks do not support a monotonic "older is always worse" or "more impressions is always worse" rule.

The strongest observed combination is the 91–180 day freshness bucket combined with moderate visibility:

* `n = 3,697`
* declining rate = `65.6%`

The same 91–180 day bucket has lower observed decline rates for good visibility (`59.8%`) and excellent visibility (`43.3%`). Within the moderate-visibility group, decline rates vary between 62.9% and 68.2% across impression sub-buckets, so impressions are not used as a continuous risk score.

I therefore use one simple baseline rule:

**91–180 days since last update AND moderate visibility → REFRESH_REVIEW**

The baseline score is binary: `1` when the rule fires and `0` otherwise. Among pages with the same score, `impressions_90d` is used only as a tie-breaker to prioritize pages with more search exposure at stake; it is not treated as a stronger decline signal.

**Reason code:** `STALE_WITH_MODERATE_VISIBILITY`

**Action:** `REFRESH_REVIEW`


In [35]:
baseline = df.copy()

baseline["score"] = (
    (baseline["freshness_tier"] == "91-180") &
    (baseline["impression_tier"] == "moderate")
).astype(int)

baseline["reason_code"] = np.where(
    baseline["score"] == 1,
    "STALE_WITH_MODERATE_VISIBILITY",
    "BASELINE_NOT_TRIGGERED"
)

baseline["action"] = np.where(
    baseline["score"] == 1,
    "REFRESH_REVIEW",
    "NO_ACTION"
)

baseline_queue = (
    baseline[
        ["content_id", "score", "reason_code", "action", "impressions_90d"]
    ]
    .sort_values(
        ["score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

baseline_queue.head(10)

,content_id,score,reason_code,action,impressions_90d
0,content_cfbb20d73437,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2999
1,content_19882bd6373d,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2999
2,content_4302c4925c0f,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2998
3,content_3344bfd6994d,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2997
4,content_ccf887ee3581,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2993
5,content_51bb0bff5aed,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2993
6,content_7fde62f5a97f,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2993
7,content_b725a6ce11a9,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2988
8,content_a0e08775e954,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2986
9,content_a34aff7561ae,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2986


In [36]:
baseline_queue["score"].value_counts()

,count
score,
0,26303
1,3697


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Rule

Among content that is stale (`freshness_tier == "91-180"`) and has moderate visibility (`impression_tier == "moderate"`), send it to refresh review. Rank triggered items by `impressions_90d` descending so that higher-exposure items are reviewed first.

### Decision fields

- **Score:** `1` when the rule triggers, otherwise `0`.
- **Reason code:** `STALE_WITH_MODERATE_VISIBILITY`
- **Action:** `REFRESH_REVIEW` when triggered, otherwise `NO_ACTION`.
- **Ranking:** `score` descending, then `impressions_90d` descending.

The score is intentionally binary: the rule answers whether an item belongs in the review queue. `impressions_90d` is used only to prioritize the queue after the rule has selected the review population.

The rule does not use `trend_direction` or `trend_pct`, which are observed outcome/label-derived fields and are therefore excluded from the decision logic.

In [37]:
# Encode the single baseline rule.

RULE_FRESHNESS = "91-180"
RULE_IMPRESSIONS = "moderate"

baseline_queue = df.copy()

baseline_queue["score"] = (
    (baseline_queue["freshness_tier"] == RULE_FRESHNESS)
    & (baseline_queue["impression_tier"] == RULE_IMPRESSIONS)
).astype(int)

baseline_queue["reason_code"] = np.where(
    baseline_queue["score"] == 1,
    "STALE_WITH_MODERATE_VISIBILITY",
    "BASELINE_NOT_TRIGGERED",
)

baseline_queue["action"] = np.where(
    baseline_queue["score"] == 1,
    "REFRESH_REVIEW",
    "NO_ACTION",
)

baseline_queue = (
    baseline_queue[
        ["content_id", "score", "reason_code", "action", "impressions_90d"]
    ]
    .sort_values(
        ["score", "impressions_90d"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

baseline_queue.head(20)

,content_id,score,reason_code,action,impressions_90d
0,content_cfbb20d73437,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2999
1,content_19882bd6373d,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2999
2,content_4302c4925c0f,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2998
3,content_3344bfd6994d,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2997
4,content_ccf887ee3581,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2993
5,content_51bb0bff5aed,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2993
6,content_7fde62f5a97f,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2993
7,content_b725a6ce11a9,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2988
8,content_a0e08775e954,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2986
9,content_a34aff7561ae,1,STALE_WITH_MODERATE_VISIBILITY,REFRESH_REVIEW,2986


In [38]:
from pathlib import Path

OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote: {OUTPUT_PATH}")
print(f"Rows: {len(baseline_queue):,}")
print(f"Triggered: {(baseline_queue['score'] == 1).sum():,}")
print(f"Not triggered: {(baseline_queue['score'] == 0).sum():,}")

Wrote: work/outputs/baseline_action_score.csv
Rows: 30,000
Triggered: 3,697
Not triggered: 26,303


In [39]:
expected = baseline_queue.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False],
).reset_index(drop=True)

assert baseline_queue.equals(expected)

print("Baseline queue checks passed.")

Baseline queue checks passed.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [41]:
top20_ids = baseline_queue.head(20)["content_id"]

top20_review = (
    df[df["content_id"].isin(top20_ids)]
    .copy()
    .sort_values("impressions_90d", ascending=False)
)

top20_review[
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "search_volume",
        "competition",
        "word_count",
        "content_age_days",
        "days_since_last_update",
        "freshness_tier",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "sessions_90d",
        "engagement_rate",
        "trend_direction",
        "trend_pct"
    ]
]

,content_id,client_id,content_type,main_intent,search_volume,competition,word_count,content_age_days,days_since_last_update,freshness_tier,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,trend_direction,trend_pct
18830,content_19882bd6373d,client_8527a891e2,keyword article,informational,30.0,0.00,3430.0,309,104,91-180,2999,9,0.30,9.0,14,0.00,down,-41.7
13381,content_cfbb20d73437,client_6208ef0f77,keyword article,informational,0.0,0.00,5603.0,144,104,91-180,2999,1,0.03,31.9,11,0.00,down,-63.5
16812,content_4302c4925c0f,client_3fdba35f04,keyword article,informational,20.0,0.00,1388.0,333,104,91-180,2998,3,0.10,18.7,8,0.00,down,-79.2
19289,content_3344bfd6994d,client_19581e27de,keyword article,commercial,10.0,0.00,NaN,329,104,91-180,2997,32,1.07,3.1,45,8.89,stable,3.0
25305,content_51bb0bff5aed,client_6208ef0f77,keyword article,informational,0.0,0.00,6363.0,299,104,91-180,2993,2,0.07,13.0,204,0.00,down,-24.5
26295,content_7fde62f5a97f,client_19581e27de,keyword article,transactional,20.0,0.93,NaN,329,104,91-180,2993,21,0.70,8.0,23,8.70,down,-39.4
13572,content_ccf887ee3581,client_4e07408562,keyword article,informational,110.0,0.00,1377.0,326,104,91-180,2993,1,0.03,5.5,3,0.00,down,-70.9
22564,content_b725a6ce11a9,client_6208ef0f77,keyword article,informational,0.0,0.00,5714.0,287,104,91-180,2988,4,0.13,29.8,58,0.00,down,-46.5
24941,content_a34aff7561ae,client_6208ef0f77,keyword article,commercial,2400.0,0.22,4937.0,236,104,91-180,2986,11,0.37,17.0,72,1.39,down,-44.9
14160,content_a0e08775e954,client_19581e27de,keyword article,transactional,10.0,0.00,NaN,313,104,91-180,2986,3,0.10,18.4,26,0.00,up,47.5


### Top-20 review

1. **content_19882bd6373d — REFRESH_REVIEW:** 104 days since update and 2,999 impressions trigger the baseline, with a 41.7% decline and a relatively strong average position of 9.0. It could be wrong if the decline has a cause other than content freshness.

2. **content_cfbb20d73437 — REFRESH_REVIEW:** 104 days since update and 2,999 impressions trigger the baseline, with a 63.5% decline. It could be wrong if the low average position (31.9) is the main cause of the traffic loss rather than stale content.

3. **content_4302c4925c0f — REFRESH_REVIEW:** 104 days since update and 2,998 impressions trigger the baseline, with a 79.2% decline. It could be wrong if the low average position (18.7) and 0.1% CTR indicate a ranking or visibility problem rather than a freshness problem.

4. **content_3344bfd6994d — REFRESH_REVIEW:** 104 days since update and 2,997 impressions trigger the baseline. It could be wrong because the page is stable, has a strong average position (3.1), and a relatively high CTR (1.07%).

5. **content_51bb0bff5aed — REFRESH_REVIEW:** 104 days since update and 2,993 impressions trigger the baseline, with a 24.5% decline. It could be wrong because the decline is relatively modest, search volume is 0, and average position is 13.0.

6. **content_7fde62f5a97f — REFRESH_REVIEW:** 104 days since update and 2,993 impressions trigger the baseline, with a 39.4% decline. It could be wrong because its engagement rate is relatively high (8.7%), suggesting that the content may still work well for users who reach it.

7. **content_ccf887ee3581 — REFRESH_REVIEW:** 104 days since update and 2,993 impressions trigger the baseline, with a 70.9% decline despite an average position of 5.5. It could be wrong if the extremely low CTR (0.03%) reflects a snippet or intent-matching problem rather than stale content.

8. **content_b725a6ce11a9 — REFRESH_REVIEW:** 104 days since update and 2,988 impressions trigger the baseline, with a 46.5% decline. It could be wrong if the very low average position (29.8) is the main explanation for the decline rather than content freshness.

9. **content_a34aff7561ae — REFRESH_REVIEW:** 104 days since update and 2,986 impressions trigger the baseline, with a 44.9% decline and 2,400 search volume. It could be wrong if the average position of 17.0 is the primary cause of the decline rather than content freshness.

10. **content_a0e08775e954 — REFRESH_REVIEW:** 104 days since update and 2,986 impressions trigger the baseline. It could be wrong because the page is up 47.5%, so refreshing it could introduce unnecessary changes to content that is improving.

11. **content_9c0238397980 — REFRESH_REVIEW:** 104 days since update and 2,984 impressions trigger the baseline, with a 53.4% decline and average position of 4.5. It could be wrong if the decline is caused by a factor unrelated to content freshness.

12. **content_53b1fe682dee — REFRESH_REVIEW:** 104 days since update and 2,977 impressions trigger the baseline, with a 52.5% decline. It could be wrong if the average position of 15.9 and other ranking factors explain the decline rather than stale content.

13. **content_9125007c00e3 — REFRESH_REVIEW:** 104 days since update and 2,977 impressions trigger the baseline, with a 41.8% decline. It could be wrong if the page's very low CTR (0.03%) reflects a search-result presentation or intent problem rather than a freshness problem.

14. **content_912bd3805fc3 — REFRESH_REVIEW:** 104 days since update and 2,975 impressions trigger the baseline. It could be wrong because the page is classified as stable despite a -17.9% trend change, making the need for an immediate refresh less clear.

15. **content_9238cb4805c1 — REFRESH_REVIEW:** 104 days since update and 2,975 impressions trigger the baseline, with a 71.8% decline. It could be wrong if the low average position (33.0) is the primary cause of the decline rather than content freshness.

16. **content_33cde508c590 — REFRESH_REVIEW:** 104 days since update and 2,973 impressions trigger the baseline, with a 77.2% decline. It could be wrong if the average position of 11.0 and very low CTR (0.1%) indicate a ranking or snippet problem instead of stale content.

17. **content_6779f2b0058c — REFRESH_REVIEW:** 104 days since update and 2,972 impressions trigger the baseline, with a 31.0% decline. It could be wrong because the decline is relatively modest and the average position is already fairly strong at 6.9.

18. **content_701f7861d94a — REFRESH_REVIEW:** 106 days since update and 2,971 impressions trigger the baseline, with a 68.8% decline. It could be wrong if the low session volume and average position of 8.5 indicate another explanation for the observed decline.

19. **content_a466b8ddcbbf — REFRESH_REVIEW:** 104 days since update and 2,969 impressions trigger the baseline, with a 51.3% decline. It could be wrong because the page has a relatively strong average position of 11.4 and the observed decline may not be caused by freshness.

20. **content_ec10b8aa565b — REFRESH_REVIEW:** 104 days since update and 2,968 impressions trigger the baseline, with a 34.2% decline. It could be wrong because the decline is relatively modest and the page already has an average position of 8.4.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The clearest weak picks are:

- **content_3344bfd6994d:** the page is stable (+3.0%), ranks very well (average position 3.1), and has a relatively strong CTR (1.07%). The refresh rule is therefore difficult to justify from the observed performance.
- **content_a0e08775e954:** the page is up 47.5%. A refresh recommendation is questionable when the observed trend is strongly positive.
- **content_51bb0bff5aed:** although it triggers the rule and is declining, the decline is relatively modest (-24.5%), search volume is 0, and average position is 13.0. The case for prioritizing a refresh is weaker than for several other candidates.

These examples show that the baseline identifies a review population rather than proving that a refresh is the correct intervention.

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.